# Sentence Similarity — Hybrid Embedding (LSA + BM25)

In [ ]:
import sys
import numpy as np
sys.path.insert(0, '.')
from main import HybridEmbedder

model = HybridEmbedder.load('models/model')
print(model.lsa)
print(model.bm25)

In [ ]:
sentence_A = "the dog is sleeping on the floor"
sentence_B = "the dog is awake and running around"
alpha = 0.5   # 0 = pure dense (semantic)  |  1 = pure sparse (keyword)

dense_A, sparse_vec_A = model.encode(sentence_A)
dense_B, sparse_vec_B = model.encode(sentence_B)

dense_cosine = float(np.dot(dense_A, dense_B))

norm_A = np.linalg.norm(sparse_vec_A)
norm_B = np.linalg.norm(sparse_vec_B)
if norm_A > 1e-12 and norm_B > 1e-12:
    sparse_cosine = float(np.dot(sparse_vec_A, sparse_vec_B) / (norm_A * norm_B))
else:
    sparse_cosine = 0.0

hybrid = (1 - alpha) * dense_cosine + alpha * sparse_cosine

print(f"A : {sentence_A!r}")
print(f"B : {sentence_B!r}")
print()
print(f"  Dense  cosine  (LSA)  : {dense_cosine:.4f}")
print(f"  Sparse cosine  (BM25) : {sparse_cosine:.4f}")
print(f"  Hybrid (alpha={alpha}) : {hybrid:.4f}   <-- final score")
print()
print("  Score guide: -1=opposite  0=unrelated  1=identical")

sparse_A = model.bm25.encode_as_dict(sentence_A)
sparse_B = model.bm25.encode_as_dict(sentence_B)
rev = {v: k for k, v in model.bm25.vocabulary.items()}
shared = sorted(
    [(rev[t], sparse_A[t], sparse_B[t]) for t in sparse_A if t in sparse_B],
    key=lambda x: -(x[1] + x[2]),
)
print()
if shared:
    print(f"  Shared keywords ({len(shared)}):")
    for term, wa, wb in shared:
        print(f"    '{term}'   A={wa:.2f}  B={wb:.2f}")
else:
    print("  No shared keywords — sparse cosine = 0, hybrid relies on dense only")

## Pearson / Spearman Sweep

In [ ]:
import csv, time
import numpy as np

CSV_PATH = r'C:\Users\AmrSaad\Desktop\VidSeek\vidseek\Transformer\data\sts-222\stsb_test.csv'

pairs = []
with open(CSV_PATH, encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        pairs.append((row['sentence1'], row['sentence2'], float(row['score'])))

print(f"Loaded {len(pairs)} STS-B pairs")

gold, dense_scores, sparse_scores = [], [], []

t0 = time.time()
for s1, s2, label in pairs:
    d1, sp1 = model.encode(s1)
    d2, sp2 = model.encode(s2)

    dc = float(np.dot(d1, d2))
    n1, n2 = np.linalg.norm(sp1), np.linalg.norm(sp2)
    sc = float(np.dot(sp1, sp2) / (n1 * n2)) if n1 > 1e-12 and n2 > 1e-12 else 0.0

    gold.append(2 * label - 1)
    dense_scores.append(dc)
    sparse_scores.append(sc)

print(f"Encoded {len(pairs)} pairs in {time.time()-t0:.1f}s")

gold   = np.array(gold)
dense  = np.array(dense_scores)
sparse = np.array(sparse_scores)

def pearson(x, y):
    x, y = x - x.mean(), y - y.mean()
    denom = np.linalg.norm(x) * np.linalg.norm(y)
    return float(np.dot(x, y) / denom) if denom > 1e-12 else 0.0

def spearman(x, y):
    rx = np.argsort(np.argsort(x)).astype(float)
    ry = np.argsort(np.argsort(y)).astype(float)
    return pearson(rx, ry)

print()
print(f"{'Mode':<22}  {'Pearson':>8}  {'Spearman':>9}")
print("-" * 44)

p_lsa  = pearson(dense, gold);  s_lsa  = spearman(dense, gold)
p_bm25 = pearson(sparse, gold); s_bm25 = spearman(sparse, gold)
print(f"  {'LSA  (dense only)':<20}  {p_lsa:>8.4f}  {s_lsa:>9.4f}")
print(f"  {'BM25 (sparse only)':<20}  {p_bm25:>8.4f}  {s_bm25:>9.4f}")
print()

best_p, best_alpha = -1, 0.5
for alpha in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    hybrid = (1 - alpha) * dense + alpha * sparse
    p = pearson(hybrid, gold)
    s = spearman(hybrid, gold)
    print(f"  {'hybrid a='+str(alpha):<20}  {p:>8.4f}  {s:>9.4f}")
    if p > best_p:
        best_p, best_alpha = p, alpha

best_hybrid = (1 - best_alpha) * dense + best_alpha * sparse
print()
print(f"  Best alpha = {best_alpha}  ->  Pearson {best_p:.4f}  Spearman {spearman(best_hybrid, gold):.4f}")

## Model Comparison — Recall@K, MRR, Spearman, Latency

Retrieval task: for each pair `(s1, s2)`, `s1` is the query and `s2` is its single positive.
All unique sentences form the corpus; self-match is excluded from ranking.

In [ ]:
import sys, csv, time
import numpy as np
sys.path.insert(0, '.')
from main import HybridEmbedder

model = HybridEmbedder.load('models/model')
print(model.lsa)
print(model.bm25)

CSV_PATH = r'C:\Users\AmrSaad\Desktop\VidSeek\vidseek\Transformer\data\sts-222\stsb_test.csv'
ALPHA = 0.9

pairs = []
with open(CSV_PATH, encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        pairs.append((row['sentence1'], row['sentence2'], float(row['score'])))

all_sents = list({s for s1, s2, _ in pairs for s in (s1, s2)})
sent2idx  = {s: i for i, s in enumerate(all_sents)}
N = len(all_sents)
print(f"Corpus: {N:,} unique sentences | {len(pairs):,} pairs")

t0 = time.time()
dense_mat, sparse_mat = [], []
for s in all_sents:
    d, sp = model.encode(s)
    dense_mat.append(d)
    sparse_mat.append(sp)

dense_mat  = np.stack(dense_mat)   # (N, d)
sparse_mat = np.stack(sparse_mat)  # (N, V)

sp_norms    = np.linalg.norm(sparse_mat, axis=1, keepdims=True)
sp_norms    = np.where(sp_norms < 1e-12, 1.0, sp_norms)
sparse_norm = sparse_mat / sp_norms
embed_time  = time.time() - t0
print(f"Embedded {N:,} sentences in {embed_time:.1f}s")

def lsa_scores(q_idx):
    return dense_mat @ dense_mat[q_idx]

def bm25_scores(q_idx):
    return sparse_norm @ sparse_norm[q_idx]

def hybrid_scores(q_idx):
    return (1 - ALPHA) * lsa_scores(q_idx) + ALPHA * bm25_scores(q_idx)

def pearson(x, y):
    x, y = x - x.mean(), y - y.mean()
    d = np.linalg.norm(x) * np.linalg.norm(y)
    return float(np.dot(x, y) / d) if d > 1e-12 else 0.0

def spearman_corr(pred, gold):
    return pearson(
        np.argsort(np.argsort(pred)).astype(float),
        np.argsort(np.argsort(gold)).astype(float),
    )

def evaluate(score_fn):
    recall = {1: 0, 5: 0, 10: 0}
    mrr       = 0.0
    pair_sims = []
    t0 = time.time()

    for s1, s2, _ in pairs:
        q_idx   = sent2idx[s1]
        pos_idx = sent2idx[s2]

        s_all = score_fn(q_idx)
        pair_sims.append(float(s_all[pos_idx]))

        s_rank        = s_all.copy()
        s_rank[q_idx] = -np.inf
        ranked        = np.argsort(-s_rank)
        rank          = int(np.where(ranked == pos_idx)[0][0]) + 1

        for k in [1, 5, 10]:
            if rank <= k:
                recall[k] += 1
        mrr += 1.0 / rank

    elapsed = time.time() - t0
    n       = len(pairs)
    gold    = np.array([2 * sc - 1 for _, _, sc in pairs])

    return {
        'R@1':        recall[1]  / n,
        'R@5':        recall[5]  / n,
        'R@10':       recall[10] / n,
        'MRR':        mrr        / n,
        'Spearman':   spearman_corr(np.array(pair_sims), gold),
        'latency_ms': elapsed / n * 1000,
    }

print()
models_to_eval = [
    ("LSA  (dense)",      lsa_scores),
    ("BM25 (sparse)",     bm25_scores),
    (f"Hybrid a={ALPHA}", hybrid_scores),
]

results = {}
for name, fn in models_to_eval:
    print(f"  Evaluating {name} ...", end="", flush=True)
    results[name] = evaluate(fn)
    print(f" done  ({results[name]['latency_ms']:.2f} ms/query)")

print()
print(f"  {'Model':<20}  {'R@1':>6}  {'R@5':>6}  {'R@10':>6}  {'MRR':>6}  {'Spearman':>9}  {'ms/q':>7}")
print("  " + "-" * 68)
for name, m in results.items():
    print(
        f"  {name:<20}  {m['R@1']:>6.4f}  {m['R@5']:>6.4f}  {m['R@10']:>6.4f}"
        f"  {m['MRR']:>6.4f}  {m['Spearman']:>9.4f}  {m['latency_ms']:>7.3f}"
    )

[LSAEncoder] loaded from models/model_lsa
LSAEncoder(n_components=128, vocab_size=48850, k=128)
Corpus: 180,953 unique sentences | 100,000 pairs


## Similarity Demo

In [ ]:
import numpy as np

def similarity(text1: str, text2: str, alpha: float = 0.9) -> float:
    d1, sp1 = model.encode(text1)
    d2, sp2 = model.encode(text2)
    dc = float(np.dot(d1, d2))
    n1, n2 = np.linalg.norm(sp1), np.linalg.norm(sp2)
    sc = float(np.dot(sp1, sp2) / (n1 * n2)) if n1 > 1e-12 and n2 > 1e-12 else 0.0
    return (1 - alpha) * dc + alpha * sc

test_pairs = [
    ("A dog is running in the park",    "A dog is playing outside"),
    ("The man is eating a sandwich",    "A person is having lunch"),
    ("A woman is singing a song",       "A lady is performing music"),
    ("The cat sat on the mat",          "A dog is running in the park"),
    ("A man is riding a bicycle",       "Someone is driving a car"),
    ("I love pizza",                    "The stock market crashed today"),
]

print(f'\n{"Sentence 1":<42} {"Sentence 2":<42} {"Sim":>8}')
print("-" * 100)

for s1, s2 in test_pairs:
    sim = similarity(s1, s2)
    bar = "#" * max(0, int((sim + 1) * 10))
    print(f"{s1:<42} {s2:<42} {sim:>8.4f}  {bar}")